# YouTube Ranking

In [62]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from tqdm.notebook import tqdm

In [2]:
url = 'https://youtube-rank.com/board/bbs/board.php?bo_table=youtube'
driver = webdriver.Chrome('chromedriver')
driver.get(url)
time.sleep(2)

- step 1,2

In [4]:
soup = BeautifulSoup(driver.page_source, 'html.parser')
trs = soup.select('tr.aos-init')
len(trs)

100

- step 3

In [10]:
# 랭크정보 가져오기
tr = trs[0]
tr.select_one('.rank').get_text().strip()

'1'

In [14]:
# 카테고리 정보 가져오기
tr.select_one('.category').get_text().strip()

'[음악/댄스/가수]'

In [15]:
# 채널명 가져오기
tr.select_one('.subject > h1 > a').get_text().strip()

In [20]:
# 구독자 & 조회수 & 비디오개수
subscriber = tr.select_one('.subscriber_cnt').get_text().strip()
view = tr.select_one('.view_cnt').get_text().strip()
video = tr.select_one('.video_cnt').get_text().strip()
print(subscriber, view, video)

8380만 286억1398만 466개


In [35]:
lines = []
for tr in trs:
    rank = tr.select_one('.rank').get_text().strip()
    category = tr.select_one('.category').get_text().strip()[1:-1]
    chennel = tr.select_one('.subject > h1 > a').get_text().strip()
    subscriber = tr.select_one('.subscriber_cnt').get_text().strip()
    view = tr.select_one('.view_cnt').get_text().strip()
    video = tr.select_one('.video_cnt').get_text().strip()
    lines.append([rank, category, chennel, subscriber, view, video])

In [36]:
df = pd.DataFrame(lines, columns = ['순위', '카테고리', '채널명','구독자수','조회수','영상수'])

In [37]:
df

,순위,카테고리,채널명,구독자수,조회수,영상수
0,1,음악/댄스/가수,BLACKPINK,8380만,286억1398만,466개
1,2,음악/댄스/가수,BANGTANTV,7300만,191억4327만,"2,086개"
2,3,음악/댄스/가수,HYBE LABELS,6950만,258억4674만,"1,065개"
3,4,음악/댄스/가수,SMTOWN,3130만,262억1822만,"4,052개"
4,5,키즈/어린이,Boram Tube Vlog [보람튜브 브이로그],2650만,110억5288만,223개
...,...,...,...,...,...,...
95,96,TV/방송,JTBC Drama,469만,50억8639만,"27,000개"
96,98,키즈/어린이,CoCosToy 꼬꼬스토이,467만,28억7649만,656개
97,97,음식/요리/레시피,Cooking tree 쿠킹트리,467만,4억6943만,"1,410개"
98,99,음식/요리/레시피,까니짱 [ G-NI : 蟹ちゃん],465만,13억5018만,719개


In [45]:
df['조회수']

0     286억1398만
1     191억4327만
2     258억4674만
3     262억1822만
4     110억5288만
        ...    
95     50억8639만
96     28억7649만
97      4억6943만
98     13억5018만
99      8억3879만
Name: 조회수, Length: 100, dtype: object

In [ ]:
format(int(view.replece('억','').replace('만', '0000')))

In [56]:
def convert_unit(s):
    s = s.replace('억','').replace('개','').replace(',','').replace('만','0000')
    return f'{int(s):,d}'

In [57]:
convert_unit('286억1398만')

'28,613,980,000'

In [58]:
# df['조회수'] = df['조회수'].apply(convert_unit)
# df['구독자수'] = df['구독자수'].apply(convert_unit
# df['영상수'] = df['영상수'].apply(convert_unit)
for column in df.columns[3:]:
    df[column] = df[column].apply(convert_unit)

In [59]:
df

,순위,카테고리,채널명,구독자수,조회수,영상수
0,1,음악/댄스/가수,BLACKPINK,"83,800,000","28,613,980,000",466
1,2,음악/댄스/가수,BANGTANTV,"73,000,000","19,143,270,000","2,086"
2,3,음악/댄스/가수,HYBE LABELS,"69,500,000","25,846,740,000","1,065"
3,4,음악/댄스/가수,SMTOWN,"31,300,000","26,218,220,000","4,052"
4,5,키즈/어린이,Boram Tube Vlog [보람튜브 브이로그],"26,500,000","11,052,880,000",223
...,...,...,...,...,...,...
95,96,TV/방송,JTBC Drama,"4,690,000","5,086,390,000","27,000"
96,98,키즈/어린이,CoCosToy 꼬꼬스토이,"4,670,000","2,876,490,000",656
97,97,음식/요리/레시피,Cooking tree 쿠킹트리,"4,670,000","469,430,000","1,410"
98,99,음식/요리/레시피,까니짱 [ G-NI : 蟹ちゃん],"4,650,000","1,350,180,000",719


In [67]:
# tqdm을 사용하여 10페이지 추출해보기
# youtubeRank.csv 저장
for i in tqdm(range(1,11)):
    sub_url = f'https://youtube-rank.com/board/bbs/board.php?bo_table=youtube&page={i}'
    driver2 = webdriver.Chrome('chromedriver')
    driver2.get(sub_url)
    time.sleep(2)
    soup2 = BeautifulSoup(driver2.page_source, 'html.parser')
    trs2 = soup2.select('tr.aos-init')
    tr2 = trs2[0]
    lines2 = []
    for tr2 in trs2:
        rank = tr2.select_one('.rank').get_text().strip()
        category = tr2.select_one('.category').get_text().strip()[1:-1]
        chennel = tr2.select_one('.subject > h1 > a').get_text().strip()
        subscriber = tr2.select_one('.subscriber_cnt').get_text().strip()
        view = tr2.select_one('.view_cnt').get_text().strip()
        video = tr2.select_one('.video_cnt').get_text().strip()
        lines2.append([rank, category, chennel, subscriber, view, video])
df2 = pd.DataFrame(lines, columns = ['순위', '카테고리', '채널명','구독자수','조회수','영상수'])
def convert_unit(s):
    s = s.replace('억','').replace('개','').replace(',','').replace('만','0000')
    return f'{int(s):,d}'
for column in df2.columns[3:]:
    df2[column] = df2[column].apply(convert_unit)

  0%|          | 0/10 [00:00<?, ?it/s]

In [69]:
df2.tail()

,순위,카테고리,채널명,구독자수,조회수,영상수
95,96,TV/방송,JTBC Drama,"4,690,000","5,086,390,000","27,000"
96,98,키즈/어린이,CoCosToy 꼬꼬스토이,"4,670,000","2,876,490,000",656
97,97,음식/요리/레시피,Cooking tree 쿠킹트리,"4,670,000","469,430,000","1,410"
98,99,음식/요리/레시피,까니짱 [ G-NI : 蟹ちゃん],"4,650,000","1,350,180,000",719
99,100,게임,EA SPORTS FIFA,"4,540,000","838,790,000",803
